# ML 벤치마크
Purpose: compare CPU-only classical ML candidates using the immutable shared validation contract.

> Warning: this is an oracle/sanity-only synthetic-data benchmark, not real-device or medical-performance evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False

device = "cpu"
ML_VIEW_ROOT = Path("/kaggle/working/goal15_ml_view")
ML_BENCHMARK_OUTPUT_ROOT = Path("/kaggle/working/goal15_ml_benchmark")
STABLE_KEYS = ("person_key", "canonical_time")
PATTERN_TARGET = "pattern_binary"
ONSET_EVENT_TARGET = "event_binary"
STAGE_TARGET = "stage_code"
STAGE_CODES = ("LOW", "MEDIUM", "HIGH", "DECREASING", "RECOVERY")
BEHAVIOR_CODES = (
    "ear_covering",
    "exit_attempt",
    "head_turn_away",
    "motion_freeze",
    "movement_reduction",
    "repetitive_body_movement",
    "repetitive_hand_movement",
    "repetitive_object_contact",
    "sustained_pressure_or_contact",
    "withdrawal_movement",
)
REQUIRED_VIEW_COLUMNS = (
    *STABLE_KEYS,
    PATTERN_TARGET,
    ONSET_EVENT_TARGET,
    "hard_negative",
    STAGE_TARGET,
    *BEHAVIOR_CODES,
)

CAUSAL_FACTORS = (
    "autonomic_arousal",
    "motor_activation",
    "cognitive_load",
    "sleep_pressure",
    "sensory_context",
    "recovery_capacity",
    "social_context",
)
ROLLING_STATISTICS = ("mean", "std", "slope")
ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
TIME_FEATURE_COLUMNS = (
    "time_sin",
    "time_cos",
    "weekday_sin",
    "weekday_cos",
    "is_awake",
)
CONTEXT_FEATURE_COLUMNS = (
    "context__sleep",
    "context__transition",
    "context__meal_context",
    "context__focused_task",
    "context__moderate_activity",
    "context__light_activity",
    "context__wake_rest",
    "context__sedentary_activity",
)
ALLOWED_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in CAUSAL_FACTORS
        for feature in (
            f"{factor}__robust_z",
            *(
                f"{factor}__{statistic}_{window_seconds}s"
                for window_seconds in ROLLING_WINDOWS_SECONDS
                for statistic in ROLLING_STATISTICS
            ),
        )
    ]
    + list(TIME_FEATURE_COLUMNS)
    + list(CONTEXT_FEATURE_COLUMNS)
)
EXACT_NON_FEATURE_COLUMNS = frozenset({
    "person_id",
    "person_key",
    "run_id",
    "dataset_id",
    "canonical_time",
    "timestamp_utc",
    "split_role",
    "context",
    "context_state",
    "event_id",
    "session_id",
    "day",
    "date",
    "source_path",
    "label_source",
    "label_confidence",
    "standard_type",
    "ood_status",
    "event_type",
    "is_target",
    "hard_negative",
    "is_hard_negative",
    "hard_negative_kind",
    "event_label",
    "label",
    "missing_block_id",
    "forecast_60s",
    PATTERN_TARGET,
    ONSET_EVENT_TARGET,
    STAGE_TARGET,
    *BEHAVIOR_CODES,
})
PREDICTION_COLUMNS = [
    "model_family",
    "model_name",
    "series_id",
    "dataset_id",
    "run_id",
    "person_key",
    "canonical_time",
    "split_role",
    "target",
    "label",
    "probability",
    "threshold",
]
METRIC_COLUMNS = [
    "model_family",
    "model_name",
    "series_id",
    "split_role",
    "target",
    "metric",
    "value",
    "support",
    "data_status",
]


## 1. Validate ML role views
The benchmark fails closed unless each role view has the hash-verified shared identity and all event, stage, and behavior targets.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _require_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or len(value) != 64:
        raise ValueError(f"invalid SHA-256 for {field}")
    try:
        int(value, 16)
    except ValueError as exc:
        raise ValueError(f"invalid SHA-256 for {field}") from exc
    return value


def verify_ml_view_manifest(view_root: Path = ML_VIEW_ROOT) -> dict[str, Any]:
    manifest_path = view_root / "view_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"missing ML view manifest: {manifest_path}")
    manifest = json.loads(manifest_path.read_text())
    if manifest.get("series_id") != SERIES_ID:
        raise ValueError("ML view manifest series_id mismatch")
    if manifest.get("data_status") != DATA_STATUS:
        raise ValueError("ML view manifest data_status mismatch")
    _require_sha256(manifest.get("source_dataset_hash"), "source_dataset_hash")
    _require_sha256(manifest.get("split_hash"), "split_hash")
    files = manifest.get("files")
    if not isinstance(files, dict) or set(files) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError("ML view manifest must declare exactly train, validation, locked_test")
    for split_role, metadata in files.items():
        if not isinstance(metadata, dict):
            raise ValueError(f"ML view metadata is invalid for {split_role}")
        path_name = metadata.get("path")
        if not isinstance(path_name, str) or Path(path_name).name != path_name:
            raise ValueError(f"ML view path is invalid for {split_role}")
        view_path = view_root / path_name
        if not view_path.is_file():
            raise FileNotFoundError(f"missing ML role view: {view_path}")
        if sha256_file(view_path) != _require_sha256(metadata.get("sha256"), path_name):
            raise ValueError(f"ML role view hash mismatch: {path_name}")
        if not isinstance(metadata.get("row_count"), int) or metadata["row_count"] < 1:
            raise ValueError(f"ML role view row_count is invalid for {split_role}")
        columns = metadata.get("columns")
        if not isinstance(columns, list) or not all(isinstance(column, str) for column in columns):
            raise ValueError(f"ML role view columns are invalid for {split_role}")
        missing = sorted(set(REQUIRED_VIEW_COLUMNS).difference(columns))
        if missing:
            raise ValueError(f"ML role view required columns missing for {split_role}: {missing}")
    return manifest


def _validate_loaded_split_contract(views: dict[str, pd.DataFrame]) -> None:
    person_sets = {
        split_role: set(frame["person_key"].astype(str))
        for split_role, frame in views.items()
    }
    counts = {split_role: len(people) for split_role, people in person_sets.items()}
    if counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(f"person count mismatch: {counts}")
    roles = list(EXPECTED_SPLIT_COUNTS)
    overlaps = [
        sorted(person_sets[roles[left]] & person_sets[roles[right]])
        for left in range(len(roles))
        for right in range(left + 1, len(roles))
    ]
    if any(overlaps):
        raise ValueError(f"person leakage across split roles: {overlaps}")


def load_ml_views(view_root: Path = ML_VIEW_ROOT) -> dict[str, pd.DataFrame]:
    manifest = verify_ml_view_manifest(view_root)
    views: dict[str, pd.DataFrame] = {}
    for split_role, metadata in manifest["files"].items():
        frame = pd.read_parquet(view_root / str(metadata["path"]))
        if len(frame) != metadata["row_count"]:
            raise ValueError(f"ML role view row count mismatch for {split_role}")
        missing = sorted(set(REQUIRED_VIEW_COLUMNS).difference(frame.columns))
        if missing:
            raise ValueError(f"ML role view required columns missing for {split_role}: {missing}")
        views[split_role] = frame.assign(split_role=split_role)
    _validate_loaded_split_contract(views)
    return views


def _validated_feature_columns(
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
) -> list[str]:
    columns = list(feature_columns)
    if not columns:
        raise ValueError("at least one feature column is required")
    missing = sorted(set(columns).difference(frame.columns))
    if missing:
        raise ValueError(f"feature columns missing: {missing}")
    unapproved = sorted(set(columns).difference(ALLOWED_FEATURE_COLUMNS))
    if unapproved:
        raise ValueError(f"unapproved columns requested as features: {unapproved}")
    nonnumeric = [
        column
        for column in columns
        if not (
            pd.api.types.is_numeric_dtype(frame[column].dtype)
            or pd.api.types.is_bool_dtype(frame[column].dtype)
        )
    ]
    if nonnumeric:
        raise ValueError(f"allowed feature columns must be numeric or bool: {sorted(nonnumeric)}")
    return columns


def _feature_matrix(frame: pd.DataFrame, feature_columns: Sequence[str]) -> np.ndarray:
    columns = _validated_feature_columns(frame, feature_columns)
    matrix = frame.loc[:, columns].to_numpy(dtype=np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError("feature matrix contains non-finite values")
    return matrix


def _fit_binary_estimator(estimator: Any, matrix: np.ndarray, target: pd.Series, name: str) -> Any:
    values = target.to_numpy(dtype=np.int8)
    if set(np.unique(values)) != {0, 1}:
        raise ValueError(f"{name} needs both binary classes")
    estimator.fit(matrix, values)
    return estimator


def _select_stage_training_rows(frame: pd.DataFrame) -> pd.Series:
    return frame[PATTERN_TARGET].eq(1)


def _select_behavior_training_rows(frame: pd.DataFrame) -> pd.Series:
    hard_negative_values = set(frame["hard_negative"].dropna().unique())
    if not hard_negative_values.issubset({0, 1, False, True}):
        raise ValueError("hard_negative must be binary")
    pattern_rows = frame[PATTERN_TARGET].eq(1)
    hard_negative_rows = frame["hard_negative"].eq(1)
    behavior_positive = frame.loc[:, list(BEHAVIOR_CODES)].eq(1).any(axis=1)
    invalid_positive = behavior_positive & ~(pattern_rows | hard_negative_rows)
    if invalid_positive.any():
        raise ValueError(
            "behavior-positive rows must be pattern or hard negative"
        )
    return pattern_rows | (hard_negative_rows & behavior_positive)


def _fit_multitask_candidate(
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
    *,
    model_name: str,
    make_estimator: Any,
) -> dict[str, Any]:
    required_labels = {
        PATTERN_TARGET,
        ONSET_EVENT_TARGET,
        "hard_negative",
        STAGE_TARGET,
        *BEHAVIOR_CODES,
    }
    missing = sorted(required_labels.difference(frame.columns))
    if missing:
        raise ValueError(f"candidate training requires columns: {missing}")
    matrix = _feature_matrix(frame, feature_columns)
    pattern_model = _fit_binary_estimator(
        make_estimator(),
        matrix,
        frame[PATTERN_TARGET],
        f"{model_name} pattern",
    )

    stage_rows = frame.loc[_select_stage_training_rows(frame)].copy()
    if stage_rows.empty:
        raise ValueError(f"{model_name} needs pattern rows for conditional stage heads")
    stage_matrix = _feature_matrix(stage_rows, feature_columns)
    stage_models = {
        stage: _fit_binary_estimator(
            make_estimator(),
            stage_matrix,
            stage_rows[STAGE_TARGET].eq(stage),
            f"{model_name} stage {stage}",
        )
        for stage in STAGE_CODES
    }

    behavior_rows = frame.loc[_select_behavior_training_rows(frame)].copy()
    if behavior_rows.empty:
        raise ValueError(f"{model_name} needs behavior decision rows")
    behavior_matrix = _feature_matrix(behavior_rows, feature_columns)
    behavior_models = {
        behavior: _fit_binary_estimator(
            make_estimator(),
            behavior_matrix,
            behavior_rows[behavior],
            f"{model_name} behavior {behavior}",
        )
        for behavior in BEHAVIOR_CODES
    }
    return {
        "model_name": model_name,
        "pattern_model": pattern_model,
        "stage_models": stage_models,
        "behavior_models": behavior_models,
    }


def fit_logistic_candidate(
    frame: pd.DataFrame, feature_columns: Sequence[str]
) -> dict[str, Any]:
    def make_estimator() -> Pipeline:
        return Pipeline(
            [
                ("scale", StandardScaler()),
                ("model", LogisticRegression(
                    class_weight="balanced", max_iter=1000, random_state=17, solver="lbfgs"
                )),
            ]
        )

    return _fit_multitask_candidate(
        frame, feature_columns, model_name="logistic_regression", make_estimator=make_estimator
    )


def _make_hgb_estimator() -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        class_weight="balanced",
        max_iter=100,
        random_state=17,
    )


def fit_hgb_candidate(frame: pd.DataFrame, feature_columns: Sequence[str]) -> dict[str, Any]:
    return _fit_multitask_candidate(
        frame,
        feature_columns,
        model_name="hist_gradient_boosting",
        make_estimator=_make_hgb_estimator,
    )


## 2. Score validation only
Threshold and champion selection use validation rows exclusively. Locked-test metrics are reported only after validation has fixed the champion.

In [ ]:
def _positive_probability(model: Any, matrix: np.ndarray) -> np.ndarray:
    classes = np.asarray(model.classes_)
    positive_index = np.flatnonzero(classes == 1)
    if len(positive_index) != 1:
        raise ValueError("binary model does not expose a positive class")
    return model.predict_proba(matrix)[:, int(positive_index[0])].astype(np.float64)


def _segments(mask: np.ndarray) -> list[tuple[int, int]]:
    padded = np.pad(mask.astype(np.int8), (1, 1))
    changes = np.diff(padded)
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1) - 1
    return list(zip(starts.tolist(), ends.tolist(), strict=True))


def _event_alert_summary(
    truth: np.ndarray,
    predicted: np.ndarray,
    person_keys: np.ndarray,
    canonical_time: np.ndarray,
) -> tuple[float, float]:
    labels = np.asarray(truth, dtype=np.int8)
    alerts = np.asarray(predicted, dtype=bool)
    people = np.asarray(person_keys)
    times = np.asarray(canonical_time)
    lengths = {len(labels), len(alerts), len(people), len(times)}
    if len(lengths) != 1:
        raise ValueError("truth, predictions, person keys, and times must have equal lengths")
    grouped = pd.DataFrame({
        "truth": labels,
        "predicted": alerts,
        "person_key": people,
        "canonical_time": times,
    })
    detected = 0
    truth_event_count = 0
    false_alerts = 0
    for person_key, person in grouped.groupby("person_key", sort=False):
        if person["canonical_time"].duplicated().any():
            raise ValueError(f"duplicate canonical_time for person {person_key}")
        if not person["canonical_time"].is_monotonic_increasing:
            raise ValueError(f"canonical_time is not ordered for person {person_key}")
        person_truth = person["truth"].to_numpy(dtype=np.int8)
        person_predicted = person["predicted"].to_numpy(dtype=bool)
        truth_events = _segments(person_truth.astype(bool))
        truth_event_count += len(truth_events)
        detected += sum(
            bool(person_predicted[start : end + 1].any())
            for start, end in truth_events
        )
        false_alerts += len(
            _segments(person_predicted & ~person_truth.astype(bool))
        )
    event_recall = detected / truth_event_count if truth_event_count else 0.0
    return event_recall, float(false_alerts)


def expected_calibration_error(truth: np.ndarray, probability: np.ndarray, bins: int = 10) -> float:
    result = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        mask = (probability >= lower) & (probability < upper if upper < 1 else probability <= upper)
        if mask.any():
            result += float(mask.mean()) * abs(float(truth[mask].mean()) - float(probability[mask].mean()))
    return result


def select_validation_threshold(
    truth: np.ndarray,
    probability: np.ndarray,
    person_keys: np.ndarray,
    canonical_time: np.ndarray,
    *,
    duration_hours: float,
) -> float:
    lengths = {
        len(truth),
        len(probability),
        len(person_keys),
        len(canonical_time),
    }
    if len(lengths) != 1:
        raise ValueError("truth, probability, person keys, and times must have equal lengths")
    thresholds = np.unique(np.asarray(probability, dtype=np.float64))
    if not len(thresholds):
        return 0.5
    scores: list[tuple[float, float, float, float]] = []
    for threshold in thresholds:
        predicted = probability >= threshold
        event_recall, false_alerts = _event_alert_summary(
            truth,
            predicted,
            person_keys,
            canonical_time,
        )
        false_alerts_per_hour = false_alerts / duration_hours if duration_hours else 0.0
        scores.append((
            float(f1_score(truth, predicted, zero_division=0)),
            event_recall,
            -false_alerts_per_hour,
            float(threshold),
        ))
    return max(scores)[3]


def compute_common_metrics(
    truth: np.ndarray,
    probability: np.ndarray,
    *,
    threshold: float,
    duration_hours: float,
    model_name: str,
    split_role: str,
    target: str,
    person_keys: np.ndarray,
    canonical_time: np.ndarray,
) -> pd.DataFrame:
    labels = np.asarray(truth, dtype=np.int8)
    scores = np.asarray(probability, dtype=np.float64)
    if len(labels) != len(scores):
        raise ValueError("truth and probability lengths must match")
    if not len(labels):
        raise ValueError("metrics require at least one row")
    predicted = scores >= threshold
    event_recall, false_alerts = _event_alert_summary(
        labels,
        predicted,
        person_keys,
        canonical_time,
    )
    false_alerts_per_hour = false_alerts / duration_hours if duration_hours else 0.0
    aucpr = float(average_precision_score(labels, scores)) if len(np.unique(labels)) == 2 else np.nan
    auroc = float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else np.nan
    metric_values = {
        "aucpr": aucpr,
        "auroc": auroc,
        "event_recall": event_recall,
        "row_recall": float(recall_score(labels, predicted, zero_division=0)),
        "row_f1": float(f1_score(labels, predicted, zero_division=0)),
        "false_alerts_per_hour": false_alerts_per_hour,
        "brier_score": float(brier_score_loss(labels, scores)),
        "ece": expected_calibration_error(labels, scores),
    }
    return pd.DataFrame(
        [
            {
                "model_family": "machine_learning",
                "model_name": model_name,
                "series_id": SERIES_ID,
                "split_role": split_role,
                "target": target,
                "metric": metric,
                "value": value,
                "support": int(labels.sum()),
                "data_status": DATA_STATUS,
            }
            for metric, value in metric_values.items()
        ],
        columns=METRIC_COLUMNS,
    )


def bootstrap_people_ci(
    frame: pd.DataFrame, *, iterations: int = 1000, random_state: int = 17
) -> dict[str, float]:
    required = {"person_key", "label", "probability"}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"bootstrap rows missing columns: {missing}")
    people = np.sort(frame["person_key"].astype(str).unique())
    if len(people) < 2 or iterations < 1:
        raise ValueError("bootstrap requires at least two people and one iteration")

    def aucpr(rows: pd.DataFrame) -> float:
        labels = rows["label"].to_numpy(dtype=np.int8)
        scores = rows["probability"].to_numpy(dtype=np.float64)
        return float(average_precision_score(labels, scores)) if len(np.unique(labels)) == 2 else np.nan

    estimate = aucpr(frame)
    rng = np.random.default_rng(random_state)
    replicates: list[float] = []
    by_person = {person: frame.loc[frame["person_key"].astype(str).eq(person)] for person in people}
    for sample in rng.choice(people, size=(iterations, len(people)), replace=True):
        value = aucpr(pd.concat([by_person[person] for person in sample], ignore_index=True))
        if np.isfinite(value):
            replicates.append(value)
    if not replicates or not np.isfinite(estimate):
        raise ValueError("bootstrap samples have no two-class AUCPR")
    lower, upper = np.quantile(replicates, [0.025, 0.975])
    return {"estimate": estimate, "lower": float(lower), "upper": float(upper)}


def select_validation_champion(metric_rows: pd.DataFrame) -> str:
    validation = metric_rows.loc[
        metric_rows["split_role"].eq("validation")
        & metric_rows["target"].eq(PATTERN_TARGET)
    ].copy()
    required = {"aucpr", "event_recall", "false_alerts_per_hour", "ece"}
    table = validation.pivot_table(
        index="model_name", columns="metric", values="value", aggfunc="first"
    )
    missing = sorted(required.difference(table.columns))
    if missing:
        raise ValueError(f"validation champion metrics missing: {missing}")
    ranked = table.reset_index().sort_values(
        ["aucpr", "event_recall", "false_alerts_per_hour", "ece", "model_name"],
        ascending=[False, False, True, True, True],
        kind="mergesort",
    )
    if ranked.empty:
        raise ValueError("validation metrics are required for champion selection")
    return str(ranked.iloc[0]["model_name"])


## 3. Emit common-schema predictions and optional W&B logs
All modeling stays behind the explicit execution gate. The optional W&B login reads a Kaggle Secret at runtime only.

In [ ]:
def _prediction_rows(
    candidate: dict[str, Any],
    frame: pd.DataFrame,
    feature_columns: Sequence[str],
    *,
    split_role: str,
    pattern_threshold: float,
) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []

    def append_target(
        selected: pd.DataFrame,
        target: str,
        labels: np.ndarray,
        model: Any,
        threshold: float,
    ) -> None:
        matrix = _feature_matrix(selected, feature_columns)
        count = len(selected)
        rows.append(pd.DataFrame({
            "model_family": ["machine_learning"] * count,
            "model_name": [candidate["model_name"]] * count,
            "series_id": [SERIES_ID] * count,
            "dataset_id": selected.get(
                "dataset_id", pd.Series("unknown", index=selected.index)
            ).to_numpy(),
            "run_id": selected.get(
                "run_id", pd.Series("unknown", index=selected.index)
            ).to_numpy(),
            "person_key": selected["person_key"].to_numpy(),
            "canonical_time": selected["canonical_time"].to_numpy(),
            "split_role": [split_role] * count,
            "target": [target] * count,
            "label": labels.astype(np.int8),
            "probability": _positive_probability(model, matrix),
            "threshold": [threshold] * count,
        }, columns=PREDICTION_COLUMNS))

    append_target(
        frame,
        PATTERN_TARGET,
        frame[PATTERN_TARGET].to_numpy(),
        candidate["pattern_model"],
        pattern_threshold,
    )
    stage_frame = frame.loc[_select_stage_training_rows(frame)]
    for stage, model in candidate["stage_models"].items():
        append_target(
            stage_frame,
            f"stage::{stage}",
            stage_frame[STAGE_TARGET].eq(stage).to_numpy(),
            model,
            0.5,
        )
    behavior_frame = frame.loc[_select_behavior_training_rows(frame)]
    for behavior, model in candidate["behavior_models"].items():
        append_target(
            behavior_frame,
            f"behavior::{behavior}",
            behavior_frame[behavior].to_numpy(),
            model,
            0.5,
        )
    return pd.concat(rows, ignore_index=True)


def _infer_feature_columns(frame: pd.DataFrame) -> list[str]:
    unapproved = [
        column
        for column in frame.columns
        if column not in ALLOWED_FEATURE_COLUMNS
        and column not in EXACT_NON_FEATURE_COLUMNS
    ]
    if unapproved:
        raise ValueError(f"unapproved columns in ML role view: {sorted(unapproved)}")
    features = [
        column
        for column in ALLOWED_FEATURE_COLUMNS
        if column in frame.columns
    ]
    return _validated_feature_columns(frame, features)


def _metrics_from_predictions(predictions: pd.DataFrame, *, duration_hours: float) -> pd.DataFrame:
    metrics = [
        compute_common_metrics(
            group["label"].to_numpy(),
            group["probability"].to_numpy(),
            threshold=float(group["threshold"].iloc[0]),
            duration_hours=duration_hours,
            model_name=str(group["model_name"].iloc[0]),
            split_role=str(group["split_role"].iloc[0]),
            target=str(target),
            person_keys=group["person_key"].to_numpy(),
            canonical_time=group["canonical_time"].to_numpy(),
        )
        for target, group in predictions.groupby("target", sort=True)
    ]
    return pd.concat(metrics, ignore_index=True)


def login_wandb_from_kaggle_secret() -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        import wandb
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception as exc:
        print(f"W&B 비활성화: {type(exc).__name__}")
        return False
    return bool(wandb.login(key=key, verify=True))


def run_ml_training() -> None:
    views = load_ml_views()
    train = views["train"]
    validation = views["validation"]
    feature_columns = _infer_feature_columns(train)
    candidates = [
        fit_logistic_candidate(train, feature_columns),
        fit_hgb_candidate(train, feature_columns),
    ]
    validation_predictions: list[pd.DataFrame] = []
    validation_metrics: list[pd.DataFrame] = []
    duration_hours = len(validation) / 3600
    for candidate in candidates:
        pattern_probability = _positive_probability(
            candidate["pattern_model"], _feature_matrix(validation, feature_columns)
        )
        threshold = select_validation_threshold(
            validation[PATTERN_TARGET].to_numpy(),
            pattern_probability,
            validation["person_key"].to_numpy(),
            validation["canonical_time"].to_numpy(),
            duration_hours=duration_hours,
        )
        predictions = _prediction_rows(
            candidate, validation, feature_columns, split_role="validation", pattern_threshold=threshold
        )
        validation_predictions.append(predictions)
        validation_metrics.append(_metrics_from_predictions(predictions, duration_hours=duration_hours))
    all_validation_predictions = pd.concat(validation_predictions, ignore_index=True)
    all_validation_metrics = pd.concat(validation_metrics, ignore_index=True)
    champion_name = select_validation_champion(all_validation_metrics)
    ML_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    all_validation_predictions.to_parquet(
        ML_BENCHMARK_OUTPUT_ROOT / "validation_predictions.parquet", index=False, compression="zstd"
    )
    all_validation_metrics.to_parquet(
        ML_BENCHMARK_OUTPUT_ROOT / "validation_metrics.parquet", index=False, compression="zstd"
    )
    if RUN_LOCKED_TEST:
        champion = next(candidate for candidate in candidates if candidate["model_name"] == champion_name)
        threshold = float(all_validation_predictions.loc[
            (all_validation_predictions["model_name"] == champion_name)
            & (all_validation_predictions["target"] == PATTERN_TARGET),
            "threshold",
        ].iloc[0])
        locked = views["locked_test"]
        locked_predictions = _prediction_rows(
            champion, locked, feature_columns, split_role="locked_test", pattern_threshold=threshold
        )
        locked_predictions.to_parquet(
            ML_BENCHMARK_OUTPUT_ROOT / "locked_test_predictions.parquet", index=False, compression="zstd"
        )
        _metrics_from_predictions(locked_predictions, duration_hours=len(locked) / 3600).to_parquet(
            ML_BENCHMARK_OUTPUT_ROOT / "locked_test_metrics.parquet", index=False, compression="zstd"
        )
    if login_wandb_from_kaggle_secret():
        import wandb

        with wandb.init(
            project="multisensor-goal15-benchmark",
            group="machine-learning",
            tags=["oracle-sanity", "mvp3", "split-24-6-6", "not-real-verified"],
        ) as run:
            run.summary["validation_champion"] = champion_name
            wandb.log({"validation_metric_rows": len(all_validation_metrics)})


## 4. Explicit execution gate
The committed notebook does not validate data, train models, evaluate locked test, or contact W&B.

In [ ]:
validated_manifest = False
if RUN_TRAINING:
    validated_manifest = bool(verify_ml_view_manifest())

if not RUN_TRAINING:
    print("학습 비활성화: RUN_TRAINING=False")
elif not validated_manifest:
    raise RuntimeError("검증된 ML view manifest가 필요합니다.")
else:
    run_ml_training()
